# 05_feature_engineering_orderflow_volatility_sessions.ipynb

This notebook generates **Order Flow (CVD), Volatility, Session**, and **Candle Pattern** features, and combines all features into the final feature matrix.

### Objectives:
1. Load the SMC/liquidity dataset.
2. Compute volume delta and Cumulative Volume Delta (CVD) features and divergences.
3. Compute volatility metrics (ATR, Parkinson, Garman-Klass) and identify volatility regimes.
4. Add session features (Asia, London, New York, and overlaps).
5. Compute candle patterns and the final confluence score.
6. Save the complete feature store and feature metadata.

## 1. Environment Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import sys
import pandas as pd

PROJECT_ROOT = "/content/drive/MyDrive/btcusdt_quant_research"
sys.path.append(f"{PROJECT_ROOT}/src")

## 2. Load Dataset

In [ ]:
from utils.config import load_global_config
from utils.io_utils import load_parquet

config = load_global_config(PROJECT_ROOT)
symbol = config.get("data", "symbol", "BTCUSDT")

ict_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_ict_smc_liquidity.parquet")
df = load_parquet(ict_path)
print(f"Loaded dataset with shape: {df.shape}")

## 3. Compute Remaining Features & Confluence

In [ ]:
from features.order_flow import compute_order_flow_features
from features.volatility import compute_volatility_features
from features.sessions import compute_session_features
from features.candle_patterns import compute_candle_features
from features.confluence import compute_confluence_score
from features.feature_registry import save_feature_metadata

# Apply feature families
df = compute_order_flow_features(df)
df = compute_volatility_features(df)
df = compute_session_features(df)
df = compute_candle_features(df)
df = compute_confluence_score(df)

print(f"Final feature matrix shape: {df.shape}")
print(df[["cvd", "volatility_parkinson", "session_london", "confluence_score"]].head())

## 4. Save Complete Feature Matrix & Metadata

In [ ]:
from utils.io_utils import save_parquet

final_feature_path = os.path.join(PROJECT_ROOT, "data", "feature_store", f"{symbol}_5m_features.parquet")
save_parquet(df, final_feature_path)
print(f"Final feature matrix saved to {final_feature_path}")

# Save metadata
save_feature_metadata(PROJECT_ROOT)

## Summary of Generated Features

The complete feature store is now ready for labeling. We generated:
1. `cvd`, `cvd_slope_5`, `cvd_slope_20`, `bullish_cvd_divergence` (Order flow)
2. `volatility_parkinson`, `volatility_gk`, `bb_bandwidth` (Volatility regimes)
3. `session_asia`, `session_london`, `session_new_york` (Time-of-day sessions)
4. `confluence_score` (Net rule-based alignment score)
5. Feature metadata saved to `data/feature_store/feature_metadata.json`.

**Next Step**: Run [06_labeling_and_event_generation.ipynb](file:///content/drive/MyDrive/btcusdt_quant_research/notebooks/06_labeling_and_event_generation.ipynb) to generate target labels and meta-labels.